In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 21:00:35 WARN Utils: Your hostname, LAPTOP-AABVIV1G, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/10 21:00:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 21:00:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Question 1 – Install Spark and PySpark

In [6]:
spark.version

'4.1.1'

In [7]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-10 21:03:15--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.224.173.162, 13.224.173.176, 13.224.173.98, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.224.173.162|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M  13.4MB/s    in 5.2s    

2026-03-10 21:03:21 (12.9 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



## Question 2 – Yellow November 2025

Read the November 2025 Yellow taxi data into a Spark DataFrame, repartition to 4 partitions, write it out as Parquet, and compute the average file size (only `.parquet` files).

Run the cell below and use the printed average size (in MB) to choose the closest multiple-choice answer.

In [5]:
from pyspark.sql import functions as F
import os

input_path = 'yellow_tripdata_2025-11.parquet'
output_dir = 'data/yellow_2025_11_repartitioned'

# Read the Parquet file
yellow_df = spark.read.parquet(input_path)

# Repartition to 4 partitions and write out as Parquet
(
    yellow_df
    .repartition(4)
    .write
    .mode('overwrite')
    .parquet(output_dir)
)

# Compute average size of Parquet files (in MB)
parquet_files = [
    os.path.join(output_dir, f)
    for f in os.listdir(output_dir)
    if f.endswith('.parquet')
]

sizes_mb = [os.path.getsize(f) / (1024 * 1024) for f in parquet_files]
avg_size_mb = sum(sizes_mb) / len(sizes_mb)

print(f'Number of parquet files: {len(parquet_files)}')
print('File sizes (MB):', [round(s, 2) for s in sizes_mb])
print('Average parquet file size (MB):', round(avg_size_mb, 2))

Number of parquet files: 4
File sizes (MB): [24.4, 24.43, 24.4, 24.42]
Average parquet file size (MB): 24.42


## Question 3 – Count records on 15th November

Count how many taxi trips started on **2025-11-15**.

Run the cell below and use the printed count to choose the correct multiple-choice answer.

In [8]:
from pyspark.sql import functions as F

# Reuse yellow_df if available, otherwise read it again
try:
    yellow_df
except NameError:
    yellow_df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

trips_15 = yellow_df.filter(
    F.to_date('tpep_pickup_datetime') == F.lit('2025-11-15')
).count()

print('Number of trips starting on 2025-11-15:', trips_15)

Number of trips starting on 2025-11-15: 162604


## Question 4 – Longest trip duration (hours)

Compute the longest trip duration in hours using the pickup and dropoff timestamps.

Run the cell below and use the printed maximum duration (in hours) to choose the closest multiple-choice answer.

In [11]:
from pyspark.sql import functions as F

# Reuse yellow_df if available, otherwise read it again
try:
    yellow_df
except NameError:
    yellow_df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Compute trip duration in hours
yellow_with_duration = yellow_df.withColumn(
    'trip_duration_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600.0
)

max_duration = yellow_with_duration.agg(F.max('trip_duration_hours').alias('max_hours')).collect()[0]['max_hours']

print('Max trip duration (hours):', max_duration)

Max trip duration (hours): 90.64666666666666


## Question 5 – Spark Web UI port

The Spark Web UI for a local application runs by default on **port 4040**.

You can verify this by running a Spark job and visiting `http://localhost:4040` in your browser while the job is active.

## Question 6 – Least frequent pickup location zone

First, download the taxi zone lookup file:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Then run the cell below to load the lookup data, join it with the Yellow November 2025 trips, and find the **least frequent pickup location zone**.

Use the printed zone name to choose the correct multiple-choice answer.

In [10]:
# download the taxi zone lookup file
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv


# Load taxi zone lookup CSV
zones_path = 'taxi_zone_lookup.csv'

zones_df = spark.read.csv(zones_path, header=True, inferSchema=True)

# Reuse yellow_df if available, otherwise read it again
try:
    yellow_df
except NameError:
    yellow_df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Join on pickup location ID to get pickup zone name
joined_df = yellow_df.join(
    zones_df,
    yellow_df.PULocationID == zones_df.LocationID,
    how='left'
)

pickup_counts = (
    joined_df
    .groupBy('Zone')
    .count()
    .orderBy('count')
)

print('Least frequent pickup zones:')
pickup_counts.show(10, truncate=False)

--2026-03-10 21:07:34--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.224.173.219, 13.224.173.98, 13.224.173.176, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.224.173.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 21:07:34 (278 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]

Least frequent pickup zones:
+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3   